# pathgrade — TCGA-HNSC feature extraction on KaggleStreams slides from GDC, encodes them with **H-optimus-0** (Apache-2.0), anddiscards each slide immediately. No local WSI storage required.**Why this notebook installs a package instead of defining classes in cells.**Kaggle worker processes (DataLoader workers, and anything spawned rather thanforked) do not inherit the notebook's `__main__` namespace, so a class definedin a cell cannot be unpickled by a worker — the classic`AttributeError: Can't get attribute '...' on <module '__main__'>`. The usualworkaround is `%%writefile` to dump a module to disk first. Installing thepackage with `pip install -e` does the same job properly: every symbol lives ina real file on `sys.path`, so workers and threads import it cleanly with noduplicated source.**Disk layout matters here.**| path | size | persists | use for ||---|---|---|---|| `/kaggle/working` | ~20 GB | yes, becomes the dataset | embeddings only || `/kaggle/tmp` | ~60 GB | no | in-flight slide downloads |Slides average ~0.94 GB and the largest is ~4 GB, so the cache **must** live on`/kaggle/tmp`. Pointing it at `/kaggle/working` fills the output quota after afew slides.

## 1 · Install

In [ ]:
# Install the package so worker processes can import it (see note above).!git clone -q https://github.com/YOUR_ORG/pathgrade.git /kaggle/tmp/pathgrade || true!pip install -q -e /kaggle/tmp/pathgradeimport pathgrade, sysprint("pathgrade", pathgrade.__version__, "| python", sys.version.split()[0])

## 2 · AcceleratorSet `PATHGRADE_ACCEL` to `xla` for TPU or `cuda` for GPU.

In [ ]:
import osos.environ["PATHGRADE_ACCEL"] = "xla"   # "cuda" on a GPU session

In [ ]:
# TPU needs torch_xla; on a GPU/CPU session this cell is a no-op.import os, torchACCEL = os.environ.get("PATHGRADE_ACCEL", "auto")   # "xla" | "cuda" | "auto"if ACCEL == "xla":    !pip install -q torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html    import torch_xla.core.xla_model as xm    print("TPU device:", xm.xla_device(), "| cores:", xm.xrt_world_size())else:    print("CUDA:", torch.cuda.is_available(),          torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")import multiprocessingprint("vCPUs:", multiprocessing.cpu_count())

## 3 · Disk

In [ ]:
import shutilfor path in ["/kaggle/working", "/kaggle/tmp"]:    total, used, free = shutil.disk_usage(path)    print(f"{path:18s} free {free/1e9:6.1f} GB / total {total/1e9:6.1f} GB")CACHE_DIR = "/kaggle/tmp/wsi_cache"      # scratch, never persistedOUT_DIR   = "/kaggle/working/features"   # this becomes your Kaggle dataset

## 4 · Size the job

In [ ]:
# Query GDC and size the job before spending any quota.!python /kaggle/tmp/pathgrade/scripts/00_plan_budget.py \    --max-patches 8000 --download-mbps 50 --num-shards 4

## 5 · Trial run (do not skip)

In [ ]:
# ALWAYS run this first. Five slides tells you the real download and encode# rates, which is the one number the budget estimate cannot guess.!python /kaggle/tmp/pathgrade/scripts/01b_stream_extract.py \    --out-dir $OUT_DIR --cache-dir $CACHE_DIR \    --device $PATHGRADE_ACCEL --limit 5 --max-patches 8000 \    --batch-size 64 --decode-workers 8

## 6 · Full shard

In [ ]:
# Full shard. Change SHARD to 0/1/2/3 in four parallel sessions.SHARD, NUM_SHARDS = 0, 4# Optional: a Discord/Slack/Telegram hook so progress reaches your phone.WEBHOOK = ""   # e.g. "https://discord.com/api/webhooks/..."!python /kaggle/tmp/pathgrade/scripts/01b_stream_extract.py \    --out-dir $OUT_DIR --cache-dir $CACHE_DIR \    --device $PATHGRADE_ACCEL \    --shard {SHARD} --num-shards {NUM_SHARDS} \    --max-patches 8000 --batch-size 64 --decode-workers 8 \    --max-hours 8 --min-free-gb 3 \    --webhook-url "{WEBHOOK}" --notify-every 25

## Watching a run that takes hours1. **Kaggle real-time logs.** Use *Save & Run All (Commit)*; the notebook viewer   streams the log while it runs, so you do not need to keep this tab open.2. **Webhook pings.** Set `WEBHOOK` above to a Discord/Slack/Telegram URL and   the run pings every 25 slides plus once at the end. This is the only option   that reaches a phone.3. **Heartbeat files.** Every shard writes   `heartbeat_shard<N>.json` next to its output after each slide, with rate and   ETA. The cell below renders all shards at once — including from a previous,   killed session, since the file persists in `/kaggle/working`.A run that stops on `--max-hours` or `--min-free-gb` exits **cleanly** and skipsalready-extracted slides next time, so resuming is just re-running the cell.

In [ ]:
from pathgrade.progress import format_heartbeatsprint(format_heartbeats(OUT_DIR))

## 7 · Verify before publishing

In [ ]:
# Sanity-check the shard before publishing it as a dataset.from pathgrade.data.io import verify_cohortfrom pathlib import Pathpids = sorted(p.stem for p in Path(OUT_DIR).glob("*.h5"))info = verify_cohort(OUT_DIR, pids)for k, v in info.items():    if k != "missing":        print(f"  {k:18s} {v}")size_gb = sum(p.stat().st_size for p in Path(OUT_DIR).glob("*.h5")) / 1e9print(f"  {'output size':18s} {size_gb:.2f} GB")